# California Housing: agrupación de viviendas

## Objetivo

Agrupar observaciones del censo de California según su ubicación (`Latitude`, `Longitude`) e ingreso medio (`MedInc`) utilizando K-Means con 6 clusters. Después entrenaremos un clasificador supervisado para reproducir las etiquetas generadas automáticamente y poder asignarlas a nuevos puntos.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from app import FEATURE_COLUMNS, build_kmeans_pipeline, load_housing_data

housing = load_housing_data()
print(f"Filas válidas: {len(housing)}")
print(f"Columnas usadas: {housing.columns.tolist()}")
display(housing.describe().T)
display(housing.isna().sum().rename("nulos").to_frame())

Filas: 500
Columnas: ['review_id', 'rating', 'review_text']
Valores nulos totales: 0
Distribución de la puntuación humana:


,count
rating,
1,12
2,18
3,38
4,72
5,360


## Exploración y preparación

Las tres variables tienen escalas distintas, por lo que K-Means se entrenará dentro de un pipeline con `StandardScaler`. El escalador se ajustará únicamente con el conjunto de entrenamiento para evitar fuga de información. La división es reproducible mediante `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    housing, test_size=0.2, random_state=42
)

kmeans_pipeline = build_kmeans_pipeline()
train_clusters = kmeans_pipeline.fit_predict(train_data[FEATURE_COLUMNS])
test_clusters = kmeans_pipeline.predict(test_data[FEATURE_COLUMNS])

train_output = train_data.copy()
test_output = test_data.copy()
train_output["cluster"] = train_clusters
test_output["cluster"] = test_clusters

print(f"Train: {len(train_output)} filas | Test: {len(test_output)} filas")
print("Tamaños de los clusters en train:")
display(train_output["cluster"].value_counts().sort_index().rename("count").to_frame())
print(f"Inercia: {kmeans_pipeline.named_steps['kmeans'].inertia_:.2f}")

Device set to use cpu


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/home/vscode/.local/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/home/vscode/.local/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 566, in _l

Modelo: nlptown/bert-base-multilingual-uncased-sentiment
Commit fijado: 8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
Salida escrita en: /workspaces/thispedrito1-machine-learning-python-template/data/processed/reviews_with_sentiment.csv


,count,percentage
sentiment_band,,
negative,47,9.4
neutral,41,8.2
positive,412,82.4


## Visualización de los clusters

Los puntos se muestran sobre el mapa de California usando longitud y latitud. El color representa el cluster asignado por K-Means y el marcador diferencia observaciones de entrenamiento y prueba.

In [ ]:
fig, axis = plt.subplots(figsize=(11, 7))
axis.scatter(
    train_output["Longitude"],
    train_output["Latitude"],
    c=train_output["cluster"],
    cmap="tab10",
    s=8,
    alpha=0.35,
    label="Train",
)
axis.scatter(
    test_output["Longitude"],
    test_output["Latitude"],
    c=test_output["cluster"],
    cmap="tab10",
    s=16,
    alpha=0.75,
    marker="x",
    label="Test",
)
axis.set_xlabel("Longitude")
axis.set_ylabel("Latitude")
axis.set_title("Clusters de California Housing")
axis.legend()
axis.grid(alpha=0.2)
plt.show()

Promedio de puntuación humana: 4.50/5 (referencia comunicada: 4.5/5)
Reseñas positivas según el modelo: 82.4%
Reseñas con 4-5 estrellas humanas: 86.4%
Lectura: una media alta de estrellas puede coexistir con lenguaje neutral o negativo; por eso se revisan ambos indicadores.


## Clasificación supervisada

Las etiquetas creadas por K-Means se utilizan como objetivo de un `RandomForestClassifier`. Este modelo no descubre una verdad externa: aprende a aproximar las decisiones del clustering para poder clasificar nuevos puntos.

In [ ]:
from app import build_classifier_pipeline

classifier_pipeline = build_classifier_pipeline()
classifier_pipeline.fit(train_data[FEATURE_COLUMNS], train_clusters)
test_predictions = classifier_pipeline.predict(test_data[FEATURE_COLUMNS])

print(classification_report(test_clusters, test_predictions, zero_division=0))
print("Matriz de confusión:")
display(pd.DataFrame(
    confusion_matrix(test_clusters, test_predictions),
    index=[f"real_{index}" for index in range(6)],
    columns=[f"pred_{index}" for index in range(6)],
))

Falsos negativos identificados: 16


,review_id,rating,predicted_stars,sentiment_band,review_text
13,14,4,1,negative,Tried Harbor House Café after seeing it recomm...
14,15,5,1,negative,Every dish was bursting with flavor. The place...
72,73,5,1,negative,Stopped by Harbor House Café for the first tim...
86,87,5,1,negative,Tried Harbor House Café after seeing it recomm...
137,138,5,2,negative,Tried Harbor House Café after seeing it recomm...
199,200,5,2,negative,Stopped by Harbor House Café for the first tim...
203,204,5,1,negative,Came to Harbor House Café for a birthday brunc...
253,254,5,1,negative,Stopped by Harbor House Café for the first tim...
269,270,5,1,negative,Stopped by Harbor House Café for the first tim...
321,322,5,1,negative,Every dish was bursting with flavor. The staff...


,review_id,rating,predicted_stars,sentiment_band,review_text
361,362,3,3,neutral,Stopped by Harbor House Café for the first tim...
73,74,5,5,positive,Stopped by Harbor House Café for the first tim...
374,375,5,5,positive,Grabbed a quick coffee at Harbor House Café th...
155,156,4,4,positive,Grabbed a quick coffee at Harbor House Café th...
104,105,5,5,positive,Grabbed a quick coffee at Harbor House Café th...
394,395,4,4,positive,The food was absolutely delicious. We waited 2...
377,378,5,5,positive,Visited Harbor House Café last weekend. Loved ...
124,125,5,5,positive,Regular customer at Harbor House Café here. Th...
68,69,5,5,positive,Visited Harbor House Café last weekend. The pl...
450,451,3,3,neutral,Regular customer at Harbor House Café here. Av...


Hipótesis de revisión: los desacuerdos suelen venir de contexto de servicio, ironía, quejas sobre esperas o elogios al personal que no se parecen a reseñas de productos.


## Conclusiones

K-Means divide el territorio y el ingreso medio en seis grupos reproducibles. La separación geográfica puede observarse en el gráfico, aunque las fronteras entre clusters son una decisión matemática y no una clasificación oficial de regiones.

El clasificador supervisado obtiene una métrica alta porque aprende a reproducir las etiquetas de K-Means. Por ello, su rendimiento no demuestra que los clusters sean verdaderos ni que representen precios o calidad de vida. Los modelos finales se guardan desde `src/app.py` en `models/kmeans_model.joblib` y `models/supervised_classifier.joblib`, junto con los CSV procesados en `data/processed/`.